# 🧲 MR Damper Lab: Telemetry Analysis & Hysteresis Modeling

This interactive notebook demonstrates how to load, process, and analyze dynamic telemetry data exported from the **[MR Damper Lab Web Application](https://modified-bouc-wen-model-simulation.ai.studio)** (or its [GitHub Pages Mirror](https://waqasmbaig.github.io/Modified-Bouc-Wen-Model-Simulation/)).

### Notebook Highlights:
- **Load Telemetry**: Ingest continuous 60 FPS CSV telemetry from the Spencer Modified Bouc-Wen simulator.
- **Energetic Metrics**: Compute energy dissipation per cycle ($W_d$), equivalent damping ($C_{eq}$), and dynamic force range.
- **Nonlinear Hysteresis Characterization**: Plot Force–Displacement ($F-x$) and Force–Velocity ($F-v$) loops.
- **FFT Spectral Analysis**: Identify higher-order odd harmonics induced by Bouc-Wen hysteretic clipping.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Styling
plt.style.use('dark_background')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.facecolor'] = '#0f172a'
plt.rcParams['figure.facecolor'] = '#070d18'

## 1. Load Exported Telemetry Data

In [ ]:
# Load sample dataset from the data/ directory
df = pd.read_csv('../data/sample_harmonic_sweep.csv')
print(f"Loaded {len(df)} rows. Columns:")
print(list(df.columns))
df.head()

## 2. Compute Physical Energy & Damping Metrics

Using standard vibration engineering formulations in SI units:
- **Dissipated Work per Cycle**: $W_d = \oint F_d \, dx$
- **Equivalent Viscous Damping**: $C_{eq} = \frac{W_d}{\pi \omega X_0^2}$

In [ ]:
time = df['time_s'].values
disp = df['displacement_m'].values
vel = df['velocity_mps'].values
force = df['totalForce_N'].values

# Extract last full cycle for steady-state evaluation
freq = 1.5 # Hz
period = 1.0 / freq
mask = time >= (time[-1] - period)

x_cyc = disp[mask]
f_cyc = force[mask]
v_cyc = vel[mask]

# Numerical integration of hysteresis loop area
W_d = np.abs(np.trapezoid(f_cyc, x_cyc))
X_0 = 0.5 * (np.max(x_cyc) - np.min(x_cyc))
omega = 2.0 * np.pi * freq
C_eq = W_d / (np.pi * omega * (X_0**2))

print("=======================================================")
print(f"Stroke Amplitude (X0)   : {X_0*1e3:.2f} mm")
print(f"Excitation Frequency (f): {freq:.2f} Hz")
print(f"Peak Damper Force (Fmax): {np.max(force):.1f} N")
print(f"Work Dissipated (Wd)    : {W_d:.3f} J/cycle")
print(f"Equiv. Damping (C_eq)   : {C_eq:.1f} N*s/m")
print("=======================================================")

## 3. Visualize Hysteresis Characterization Loops

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Force vs. Displacement
ax1.plot(x_cyc * 1e3, f_cyc, color='#38bdf8', lw=2.2, label='$F_d - x$')
ax1.fill(x_cyc * 1e3, f_cyc, color='#0284c7', alpha=0.2, label=f'$W_d = {W_d:.2f}$ J')
ax1.set_title('Force vs. Displacement Loop', color='#f8fafc', fontsize=12, fontweight='bold')
ax1.set_xlabel('Displacement $x$ (mm)', color='#94a3b8')
ax1.set_ylabel('Damping Force $F_d$ (N)', color='#94a3b8')
ax1.grid(True, color='#1e293b', ls='--')
ax1.legend(loc='upper left')

# Force vs. Velocity
ax2.plot(v_cyc, f_cyc, color='#34d399', lw=2.2, label='$F_d - v$')
ax2.set_title('Force vs. Velocity Characteristics', color='#f8fafc', fontsize=12, fontweight='bold')
ax2.set_xlabel('Velocity $\\dot{x}$ (m/s)', color='#94a3b8')
ax2.set_ylabel('Damping Force $F_d$ (N)', color='#94a3b8')
ax2.grid(True, color='#1e293b', ls='--')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 4. FFT Frequency Spectrum Analysis

Examine the harmonic spectrum of the damper output force under pure sinusoidal velocity excitation. Nonlinear Bouc-Wen yielding creates distinct odd-harmonic peaks (3rd, 5th, 7th harmonics).

In [ ]:
# Compute FFT
N = len(force)
dt = np.mean(np.diff(time))
fft_vals = np.fft.rfft(force - np.mean(force))
fft_freqs = np.fft.rfftfreq(N, d=dt)
fft_mag = np.abs(fft_vals) * 2.0 / N

plt.figure(figsize=(10, 4.5))
plt.plot(fft_freqs, fft_mag, color='#f472b6', lw=1.8)
plt.xlim(0, 25)
plt.title('Damper Force FFT Spectrum (Harmonic Distortion)', color='#f8fafc', fontsize=12, fontweight='bold')
plt.xlabel('Frequency (Hz)', color='#94a3b8')
plt.ylabel('Force Amplitude (N)', color='#94a3b8')
plt.grid(True, color='#1e293b', ls='--')

# Mark fundamental and harmonics
for k in [1, 3, 5, 7]:
    f_k = k * freq
    plt.axvline(f_k, color='#fbbf24', ls=':', alpha=0.7)
    plt.text(f_k + 0.2, np.max(fft_mag)*0.8/(k**0.6), f'{k}f0 ({f_k:.1f} Hz)', color='#fbbf24', fontsize=9)

plt.tight_layout()
plt.show()